# 01 - Data Preparation (Kaggle Credit Card Transactions)

This notebook implements Phase 1 (Data Preparation) for:

- Dataset: https://www.kaggle.com/datasets/priyamchoksi/credit-card-transactions-dataset
- Goals: ingest raw CSV in Spark, clean invalid/missing/duplicate records, standardize schema, and write Parquet for reuse in later phases.

It is designed to run in both:

- Local environment (from project root)
- Google Colab (after cloning repo)

In [ ]:
# If needed (Colab/local), install dependencies:
# !pip install -r requirements.txt

from pathlib import Path
import sys

# Ensure project root is importable whether notebook runs from repo root or notebooks/.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    LongType,
)

from src.pipeline import (
    create_spark_session,
    normalize_column_names,
    standardize_schema,
    clean_transactions,
)

print(f"Imports loaded. Project root: {project_root}")

## Dataset download (Kaggle)

Use one of the following methods:

1. **Manual download** from Kaggle URL and place CSV in `data/raw/`
2. **Kaggle API** (requires `kaggle.json` credentials)

Example API command (run in terminal/Colab):

```bash
kaggle datasets download -d priyamchoksi/credit-card-transactions-dataset -p data/raw --unzip
```

After download, ensure at least one CSV file exists in `data/raw/`.

In [ ]:
# Resolve project root safely from either repo root or notebooks/ execution context.
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        f"Could not locate project root from current directory: {cwd}"
    )

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

csv_candidates = sorted(raw_dir.glob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(
        f"No CSV files found in: {raw_dir}. Download dataset and place CSV there."
    )

input_csv = csv_candidates[0]
output_parquet = processed_dir / "transactions_parquet"

print(f"Current working directory: {cwd}")
print(f"Resolved project root: {project_root}")
print(f"Using input CSV: {input_csv}")
print(f"Output Parquet path: {output_parquet}")

In [ ]:
# Explicit schema for common columns in this dataset family.
# Unknown or additional columns can still be handled by schema normalization.
raw_schema = StructType([
    StructField("Unnamed: 0", LongType(), True),
    StructField("trans_date_trans_time", StringType(), True),
    StructField("cc_num", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", DoubleType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("city_pop", LongType(), True),
    StructField("job", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", LongType(), True),
    StructField("merch_lat", DoubleType(), True),
    StructField("merch_long", DoubleType(), True),
    StructField("is_fraud", IntegerType(), True),
    StructField("merch_zipcode", StringType(), True),
])

spark = create_spark_session(app_name="CreditCardDataPrepNotebook")

raw_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("mode", "DROPMALFORMED")
    .schema(raw_schema)
    .load(str(input_csv))
)

if "Unnamed: 0" in raw_df.columns:
    raw_df = raw_df.drop("Unnamed: 0")

print("Raw row count:", raw_df.count())
raw_df.printSchema()
raw_df.show(5, truncate=False)

In [ ]:
normalized_df = normalize_column_names(raw_df)
standardized_df = standardize_schema(normalized_df)
cleaned_df = clean_transactions(standardized_df)

print("Cleaned row count:", cleaned_df.count())
cleaned_df.printSchema()
cleaned_df.select("transaction_id", "transaction_ts", "amount", "is_fraud").show(10, truncate=False)

In [ ]:
(
    cleaned_df.write
    .mode("overwrite")
    .parquet(str(output_parquet))
)

print(f"Parquet written to: {output_parquet}")

# Lightweight quality checks
print("Null amounts:", cleaned_df.filter(cleaned_df.amount.isNull()).count() if "amount" in cleaned_df.columns else "N/A")
print("Duplicate transaction_ids:", cleaned_df.groupBy("transaction_id").count().filter("count > 1").count())

spark.stop()
print("Spark session stopped.")